# Koconi 3D生成サーバー (GPU)
ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択してから実行

全セルを順番に実行してください。セル3にngrokのURLが表示されます。
そのURLを `services/api/.env` の `AI_3D_BASE_URL` に設定し、
`docker compose restart api` を実行してください。

In [ ]:
# セル1: 依存パッケージのインストール
!pip install -q fastapi uvicorn python-multipart torch
!pip install -q git+https://github.com/openai/shap-e.git
!pip install -q rembg

In [ ]:
# セル2: 3D生成サーバーのコードを書き出す
server_code = '''
import io
import os
import uuid
import threading
from concurrent.futures import ThreadPoolExecutor

import numpy as np
from PIL import Image
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
from fastapi.staticfiles import StaticFiles

app = FastAPI()

os.makedirs("generated", exist_ok=True)
app.mount("/generated", StaticFiles(directory="generated"), name="generated")

_jobs: dict[str, dict] = {}
_jobs_lock = threading.Lock()
_executor = ThreadPoolExecutor(max_workers=1)

_shap_e_lock = threading.Lock()
_shap_e_loaded = False
_shap_e_xm = None
_shap_e_model = None
_shap_e_diffusion = None

def _load_shap_e():
    global _shap_e_loaded, _shap_e_xm, _shap_e_model, _shap_e_diffusion
    with _shap_e_lock:
        if _shap_e_loaded:
            return
        import torch
        from shap_e.diffusion.gaussian_diffusion import diffusion_from_config
        from shap_e.models.download import load_config, load_model
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Loading Shap-E on {device}...")
        _shap_e_xm = load_model("transmitter", device=device)
        _shap_e_model = load_model("image300M", device=device)
        _shap_e_diffusion = diffusion_from_config(load_config("diffusion"))
        _shap_e_loaded = True
        print("Shap-E loaded.")

def _remove_background(image_bytes: bytes):
    from rembg import remove as rembg_remove
    img_rgba = Image.open(io.BytesIO(image_bytes)).convert("RGBA")
    img_no_bg = rembg_remove(img_rgba)
    background = Image.new("RGBA", img_no_bg.size, (255, 255, 255, 255))
    background.paste(img_no_bg, mask=img_no_bg.split()[3])
    return background.convert("RGB")

def _decode_latent_mesh(xm, latent):
    import torch
    from shap_e.models.nn.camera import DifferentiableCameraBatch, DifferentiableProjectiveCamera
    from shap_e.util.collections import AttrDict
    from shap_e.models.transmitter.base import Transmitter
    origins, xs, ys, zs = [], [], [], []
    for theta in np.linspace(0, 2 * np.pi, num=20):
        z = np.array([np.sin(theta), np.cos(theta), -0.5])
        z /= np.sqrt(np.sum(z**2))
        origin = -z * 4
        x = np.array([np.cos(theta), -np.sin(theta), 0.0])
        y = np.cross(z, x)
        origins.append(origin); xs.append(x); ys.append(y); zs.append(z)
    cameras = DifferentiableCameraBatch(
        shape=(1, len(xs)),
        flat_camera=DifferentiableProjectiveCamera(
            origin=torch.from_numpy(np.stack(origins)).float().to(latent.device),
            x=torch.from_numpy(np.stack(xs)).float().to(latent.device),
            y=torch.from_numpy(np.stack(ys)).float().to(latent.device),
            z=torch.from_numpy(np.stack(zs)).float().to(latent.device),
            width=2, height=2, x_fov=0.7, y_fov=0.7,
        ),
    )
    with torch.no_grad():
        decoded = xm.renderer.render_views(
            AttrDict(cameras=cameras),
            params=(xm.encoder if isinstance(xm, Transmitter) else xm).bottleneck_to_params(latent[None]),
            options=AttrDict(rendering_mode="stf", render_with_direction=False),
        )
    return decoded.raw_meshes[0]

def _run_job(job_id: str, image_bytes: bytes, base_url: str):
    with _jobs_lock:
        _jobs[job_id]["status"] = "processing"
    try:
        import torch
        from shap_e.diffusion.sample import sample_latents
        _load_shap_e()
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        img = _remove_background(image_bytes)
        latents = sample_latents(
            batch_size=1,
            model=_shap_e_model,
            diffusion=_shap_e_diffusion,
            guidance_scale=3.0,
            model_kwargs=dict(images=[img]),
            progress=False,
            clip_denoised=True,
            use_fp16=True,
            use_karras=True,
            karras_steps=32,
            sigma_min=1e-3,
            sigma_max=160,
            s_churn=0,
        )
        glb_path = f"generated/{job_id}.glb"
        for latent in latents:
            t = _decode_latent_mesh(_shap_e_xm, latent).tri_mesh()
            with open(glb_path, "wb") as f:
                t.write_glb(f)
            break
        model_url = f"{base_url}/generated/{job_id}.glb"
        with _jobs_lock:
            _jobs[job_id]["status"] = "done"
            _jobs[job_id]["model_url"] = model_url
        print(f"Job {job_id} done: {model_url}")
    except Exception as e:
        import traceback; traceback.print_exc()
        with _jobs_lock:
            _jobs[job_id]["status"] = "failed"
            _jobs[job_id]["error"] = str(e)

@app.post("/jobs")
async def create_job(file: UploadFile = File(...), taste: str = Form(default="lowpoly")):
    image_bytes = await file.read()
    base_url = os.environ.get("PUBLIC_URL", "http://localhost:8000")
    job_id = str(uuid.uuid4())
    with _jobs_lock:
        _jobs[job_id] = {"status": "pending", "model_url": "", "error": ""}
    _executor.submit(_run_job, job_id, image_bytes, base_url)
    return JSONResponse({"job_id": job_id})

@app.get("/jobs/{job_id}")
async def get_job(job_id: str):
    with _jobs_lock:
        job = _jobs.get(job_id)
    if job is None:
        return JSONResponse({"status": "not_found", "model_url": "", "error": ""}, status_code=404)
    return JSONResponse({"status": job["status"], "model_url": job.get("model_url", ""), "error": job.get("error", "")})
'''

with open('serve_3d.py', 'w') as f:
    f.write(server_code)
print('serve_3d.py を書き出しました')

In [ ]:
# セル3: localhost.run でトンネルを開く（SSHのみ、追加インストール不要）
import subprocess, time, re, os

# uvicorn をバックグラウンドで起動
server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "serve_3d:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)
print("uvicorn started")

# localhost.run トンネルを起動
tunnel_proc = subprocess.Popen(
    ["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30",
     "-R", "80:localhost:8000", "nokey@localhost.run"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

public_url = None
for _ in range(60):
    line = tunnel_proc.stdout.readline().decode(errors="ignore")
    print(line, end="")
    m = re.search(r"https://[\w.-]+\.lhr\.life", line)
    if not m:
        m = re.search(r"https://[\w.-]+\.localhost\.run", line)
    if m:
        public_url = m.group(0)
        break

os.environ["PUBLIC_URL"] = public_url or ""
print("")
print("=== localhost.run Tunnel URL ===")
print(public_url)
print("services/api/.env の AI_3D_BASE_URL を以下に設定:")
print("AI_3D_BASE_URL=" + (public_url or ""))
print("設定後: docker compose restart api")


In [ ]:
# セル4: サーバーのログを確認（任意）
# このセルは随時実行してジョブの進捗を確認できます
import subprocess
result = subprocess.run(['curl', '-s', 'http://localhost:8000/jobs/test'], capture_output=True, text=True)
print(result.stdout)